# Asian options and variance reduction

This notebook compares European and arithmetic-average Asian calls, measures antithetic variance reduction, and validates finite-difference Greeks.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from mc_options.asian_benchmarks import geometric_asian_price
from mc_options.black_scholes import black_scholes_delta, black_scholes_gamma, black_scholes_vega
from mc_options.greeks import bump_size_sensitivity, finite_difference_delta, finite_difference_gamma, finite_difference_vega, pathwise_delta
from mc_options.models import MarketParameters, OptionParameters, SimulationParameters
from mc_options.pricer import price_option
from mc_options.variance_reduction import compare_antithetic_sampling, compare_variance_reduction_methods

Asian options require full simulated paths because the payoff depends on an average of monitored prices. This project averages simulated monitoring dates after time zero by default.

In [2]:
rows = []
for sigma in [0.10, 0.20, 0.40]:
    for maturity in [0.5, 1.0, 2.0]:
        for steps in [12, 52, 252]:
            market = MarketParameters(spot=100, rate=0.03, volatility=sigma, maturity=maturity)
            sim = SimulationParameters(num_paths=50_000, num_steps=steps, seed=42)
            asian_option = OptionParameters(strike=100, option_type='call', payoff_type='asian')
            european = price_option(market, OptionParameters(strike=100, option_type='call'), sim)
            asian = price_option(market, asian_option, sim)
            geometric = geometric_asian_price(market, asian_option, num_monitoring_dates=steps)
            rows.append({
                'sigma': sigma,
                'maturity': maturity,
                'monitoring_steps': steps,
                'european_call': european.price,
                'arithmetic_asian_call': asian.price,
                'geometric_asian_call': geometric,
                'asian_discount_pct': 100 * (european.price - asian.price) / european.price,
            })

asian_results = pd.DataFrame(rows)
asian_results.query('sigma == 0.2 and maturity == 1.0')

,sigma,maturity,monitoring_steps,european_call,arithmetic_asian_call,geometric_asian_call,asian_discount_pct
12,0.2,1.0,12,9.421739,5.631835,5.435333,40.225098
13,0.2,1.0,52,9.421739,5.351171,5.166873,43.203992
14,0.2,1.0,252,9.421739,5.293007,5.103063,43.821329


In [3]:
market = MarketParameters(spot=100, rate=0.03, volatility=0.20, maturity=1.0)
option = OptionParameters(strike=100, option_type='call')
sim = SimulationParameters(num_paths=500_000, num_steps=1, seed=42)

compare_antithetic_sampling(market, option, sim)

,method,price,standard_error,ci_low,ci_high,runtime_seconds,num_paths,effective_sample_size,payoff_variance,estimator_variance,variance_reduction_pct
0,standard,9.414024,0.019996,9.374832,9.453215,0.017853,500000,500000,199.915142,0.000400,0.000000
1,antithetic,9.418259,0.014924,9.389008,9.447511,0.030325,500000,250000,55.683767,0.000223,44.292597


In [4]:
compare_variance_reduction_methods(market, option, sim)[[
    'method',
    'price',
    'standard_error',
    'estimator_variance',
    'variance_reduction_pct',
]]

,method,price,standard_error,estimator_variance,variance_reduction_pct
0,standard,9.414024,0.019996,0.000400,0.000000
1,antithetic,9.418259,0.014924,0.000223,44.292597
2,moment_matching,9.411992,0.019984,0.000399,0.122448
3,sobol,9.413402,0.019956,0.000398,0.401530
4,control_variate,9.418073,0.008210,0.000067,83.141813


In [5]:
greek_sim = SimulationParameters(num_paths=250_000, num_steps=1, seed=77)
pathwise = pathwise_delta(market, option, greek_sim)

pd.DataFrame([
    {'greek': 'delta', 'method': 'finite_difference', 'mc': finite_difference_delta(market, option, greek_sim), 'black_scholes': black_scholes_delta(market, option)},
    {'greek': 'delta', 'method': 'pathwise', 'mc': pathwise.value, 'black_scholes': black_scholes_delta(market, option)},
    {'greek': 'gamma', 'method': 'finite_difference', 'mc': finite_difference_gamma(market, option, greek_sim), 'black_scholes': black_scholes_gamma(market, option)},
    {'greek': 'vega', 'method': 'finite_difference', 'mc': finite_difference_vega(market, option, greek_sim), 'black_scholes': black_scholes_vega(market, option)},
])

,greek,method,mc,black_scholes
0,delta,finite_difference,0.598410,0.598706
1,delta,pathwise,0.598649,0.598706
2,gamma,finite_difference,0.019453,0.019333
3,vega,finite_difference,38.711259,38.666812


In [6]:
bump_size_sensitivity(
    market,
    option,
    SimulationParameters(num_paths=100_000, num_steps=1, seed=77),
)

,greek,bump,estimate,black_scholes,abs_error
0,delta,0.250,0.597945,0.598706,0.000762
1,gamma,0.250,0.019501,0.019333,0.000168
2,delta,0.500,0.597837,0.598706,0.000869
3,gamma,0.500,0.019582,0.019333,0.000248
4,delta,1.000,0.597735,0.598706,0.000972
5,gamma,1.000,0.019616,0.019333,0.000283
6,delta,2.000,0.597463,0.598706,0.001243
7,gamma,2.000,0.019664,0.019333,0.000330
8,vega,0.005,38.454612,38.666812,0.212200
9,vega,0.010,38.454164,38.666812,0.212648
